# Situation

We have a large number of Pantheon+SH0ES SNIa that need to be matched to host galaxies and uploaded to the BLAST server.  

👉 No need to involve the TNS server here — we can just work directly with our Pantheon+SH0ES data and BLAST. All the Pantheon_SH0ES that were unmatched (post TNS server upload) can be uploaded
just with name ra dec and redshift. 

---

### Conceptual reasoning

- **zHEL (heliocentric redshift):**  
  This is the directly measured spectroscopic redshift of the supernova, expressed in the heliocentric frame (corrected only for the Earth’s orbital motion around the Sun).  
  ➝ Think of it as the *raw measured redshift* from the transient spectrum.

- **zCMB:**  
  The same redshift, but transformed into the frame of the cosmic microwave background dipole (i.e., correcting for our solar system’s bulk motion relative to the CMB).  
  ➝ This is important for **cosmological analyses** (e.g., Hubble diagrams), but not for identifying or cross-matching a transient.

- **zHD:**  
  A further **Hubble-diagram adjusted redshift**, where peculiar velocity models and flow corrections are applied.  
  ➝ This is highly analysis-specific and not a directly observed value.

---

### For BLAST uploads

When uploading to BLAST (or any tool just meant to identify the transient itself via its spectrum and coordinates), the relevant property is the **observed, heliocentric redshift**:  

👉 **Use `zHEL` as the transient’s observational redshift.**  

Other redshifts (`zCMB`, `zHD`, host galaxy redshifts) are essential for cosmology, but they are not intrinsic measurements of the supernova spectrum itself.  

So it turns out that there are no RA and DEC measurements for the SNIa that were part of the DES data release and we must look at the DES .FITS files in order to open stuff up. 

In [ ]:
import pandas as pd
import os

In [ ]:
# Input (unmatched Pantheon+SH0ES file)
csv_in = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_BLAST_umatched.csv"

# Output (formatted subset file)
csv_out = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_cleaned_for_BLAST.csv"

# Load
df = pd.read_csv(csv_in)

# Pick a name/ID column — ZZTF has "ztfname"
# Prefer iau_name; fallback to ztfname when iau_name is missing/blank
if "iau_name" in df.columns and "ztfname" in df.columns:
    # Making sure that the iau names are not null and not just white spaces. 
    df["preferred_name"] = df["iau_name"].where(
        df["iau_name"].notna() & (df["iau_name"].astype(str).str.strip() != ""),
        df["ztfname"]
    )
    name_col = "preferred_name"
elif "iau_name" in df.columns:
    name_col = "iau_name"
elif "ztfname" in df.columns:
    name_col = "ztfname"
else:
    raise ValueError("Neither 'iau_name' nor 'ztfname' columns found.")

# Build new DataFrame with desired structure
out = pd.DataFrame({
    "Name": df[name_col],
    "RA": df["ra"],
    "DEC": df["dec"],
    "redshift": df["redshift"],
    "TYPE": "None"   # literal string
})

# Save
out.to_csv(csv_out, index=False)

print(f"Saved formatted file → {csv_out}")

Saved formatted file → /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_cleaned_for_BLAST.csv


In [13]:

# Input is the single cleaned file you just created
in_path  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_cleaned_for_BLAST.csv"
out_dir  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_blast_chunks"

# Make sure output directory exists
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(in_path)

# Split into chunks of 200 rows each
chunk_size = 200
n_chunks = (len(df) + chunk_size - 1) // chunk_size

for i in range(n_chunks):
    start = i * chunk_size
    end   = start + chunk_size
    chunk = df.iloc[start:end]
    out_path = os.path.join(out_dir, f"Pantheon+SH0ES_cleaned_for_BLAST_part{i+1}.csv")
    chunk["TYPE"] = "None"  # Ensure TYPE column is set to "None"
    chunk.to_csv(out_path, index=False)
    print(f"Saved {out_path} with {len(chunk)} rows")

print(f"Done! Total chunks: {n_chunks}")

Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_blast_chunks/Pantheon+SH0ES_cleaned_for_BLAST_part1.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_blast_chunks/Pantheon+SH0ES_cleaned_for_BLAST_part2.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data_ZTF/ZTF_blast_chunks/Pantheon+SH0ES_cleaned_for_BLAST_part3.csv with 70 rows
Done! Total chunks: 3


/var/folders/71/hv72gkrs7g59ty6664549kjr0000gr/T/ipykernel_35072/1914874511.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk["TYPE"] = "None"  # Ensure TYPE column is set to "None"
